# Joining Multiple Seasons: WTA Matches 2010–2023

We have 14 years of match data spread across separate CSV files. This notebook consolidates
them into a single player-season dataset — the input for the regression model in the next notebook.

Data: [JeffSackmann/tennis_wta](https://github.com/JeffSackmann/tennis_wta) · CC BY-NC-SA 4.0

In [ ]:
import pandas as pd
import glob

from errors import check_multiple_choice, get_notebook_logger, run_guarded

logger = get_notebook_logger()

## [Notebook Helpers]

This notebook imports helper functions from `errors.py`:
- `get_notebook_logger(...)` sets up consistent logging output.
- `check_multiple_choice(...)` validates MCQ responses with clear feedback.
- `run_guarded(...)` wraps risky steps and prints a learner-friendly error message before re-raising.

You can complete exercises without editing `errors.py`, but if an import fails, check that `errors.py` exists in the project root.

## [ASIDE] Why one file per year?

This is a common pattern in data engineering: partitioning data by time period keeps individual
files manageable and makes incremental updates straightforward (add a new year, don't rewrite
everything). The `glob` module lets us discover files by pattern so we're not hardcoding filenames.

## Exercise 1: Find the Match Files

**Which glob pattern finds exactly the WTA match files (excluding qualifying/ITF files)?**

- A) `"data/*.csv"` — matches all CSVs in data/, including non-match files
- B) `"data/wta_matches_*.csv"` — matches match files AND qualifying/ITF files (e.g. wta_matches_qual_itf_2023.csv)
- C) `"data/wta_matches_[0-9]*.csv"` — matches only files where the character after the last underscore is a digit (year files only) ✓
- D) `"data/**/*.csv"` — recurses into subdirectories, overkill here

Set `answer` below.

In [ ]:
# TODO: set `answer` to the correct option letter for the file-matching pattern.
answer = "C"

options = {
    "A": "data/*.csv",
    "B": "data/wta_matches_*.csv",
    "C": "data/wta_matches_[0-9]*.csv",
    "D": "data/**/*.csv",
}

check_multiple_choice(
    answer=answer,
    options=options,
    is_correct=lambda selected, choices: "[0-9]*" in choices[selected],
    exercise_label="JOIN_DATA Exercise 1",
    incorrect_message="Close, but choose the pattern that targets year-numbered files only.",
    logger=logger,
)
logger.info("JOIN_DATA Exercise 1 completed.")

In [ ]:
pattern = "data/wta_matches_[0-9]*.csv"

# TODO: assign a glob callable (e.g., glob.glob), then call it with pattern.
glob_fn = glob.glob

all_files = run_guarded(
    step_label="JOIN_DATA Exercise 2",
    action=lambda: glob_fn(pattern),
    user_error_message="Could not discover files. Check that glob_fn is callable and pattern is valid.",
    logger=logger,
)

files = [f for f in all_files if any(str(yr) in f for yr in range(2010, 2024))]
files = sorted(files)
logger.info("Discovered %d files for 2010-2023.", len(files))
print(f"Found {len(files)} files")
print(files[:3], "...")
logger.info("JOIN_DATA Exercise 2 file discovery completed.")

## [ASIDE] Loops vs copy-paste

If you've ever applied the same transformation across 14 Excel sheets manually, a loop is Python's
answer. Same logic, none of the repetition. The list we're building (`dfs`) collects all the
DataFrames before we concatenate — more efficient than concatenating inside the loop.

## Exercise 2: Load and Concatenate

Loop over the files, load each one, collect into a list, then concatenate.

In [ ]:
dfs = []
# TODO: write the loop to read each file into `dfs`, then set concat inputs below.
# Step 1: write a for-loop: for filepath in files:
#         (Steps 2 and 3 go inside the loop body - indent them)
for filepath in files:
    # Step 2: read each CSV with pd.read_csv(filepath, low_memory=False)
    df = pd.read_csv(filepath, low_memory=False)
    # Step 3: append the resulting DataFrame to dfs
    dfs.append(df)


# Step 4: choose a concat callable and ignore_index setting, then combine dfs.
concat_fn = pd.concat
use_ignore_index = True
df = run_guarded(
    step_label="JOIN_DATA Exercise 2",
    action=lambda: concat_fn(dfs, ignore_index=use_ignore_index),
    user_error_message="Could not concatenate file DataFrames. Check concat_fn, dfs, and ignore_index value.",
    logger=logger,
)
print(f"Combined shape: {df.shape}")
logger.info("JOIN_DATA Exercise 2 concatenation completed with shape %s.", df.shape)

### If This Fails, Check

- `glob_fn` and `concat_fn` are callables.
- You discovered files before concatenating (the `files` list is not empty).
- You ran cells in order so `files` and `dfs` exist.
- The `data/` folder is present in the project root.

## [ASIDE] `ignore_index=True`

Without it, `pd.concat` preserves each file's original row index — giving you duplicate index
values (multiple rows all labelled 0, 1, 2...). `ignore_index=True` resets to a clean
sequential index across the combined DataFrame. Usually what you want after a concat.

## Exercise 3: Inspect the Combined Data

Use methods you already practiced in EDA to inspect the concatenated DataFrame shape and identify columns with the highest missingness.

In [ ]:
def _exercise_3_inspect():
    # TODO: Step 1 - store the DataFrame shape in shape_value
    shape_value = ...
    print(f"Shape: {shape_value}")

    # TODO: Step 2 - compute missing values per column and sort descending into missing
    missing = ...

    # TODO: Step 3 - print the top 10 columns by missingness
    print(f"\nTop 10 columns by missingness:\n{missing.head(10)}")

run_guarded(
    step_label="JOIN_DATA Exercise 3",
    action=_exercise_3_inspect,
    user_error_message="Could not inspect the combined DataFrame. Check your shape and missingness calculations.",
    logger=logger,
)
logger.info("JOIN_DATA Exercise 3 completed.")

## Exercise 4: Filter and Select

Keep only rows with non-null values in the key stat columns.
Then select the columns we need for feature engineering.

In [ ]:
STAT_COLS = ["w_ace", "w_svpt", "w_1stIn", "w_df", "l_ace", "l_svpt", "l_1stIn", "l_df"]
# TODO: Step 1 - assign the DataFrame dropna method (unbound) to dropna_fn
# so you pass df as self when calling it.
dropna_fn = ...
df = run_guarded(
    step_label="JOIN_DATA Exercise 4",
    action=lambda: dropna_fn(df, subset=STAT_COLS),
    user_error_message="Could not drop rows with null stat columns. Check dropna_fn and STAT_COLS.",
    logger=logger,
)

KEEP_COLS = [
    "tourney_date", "surface",
    "winner_name", "winner_rank", "winner_rank_points",
    "w_ace", "w_svpt", "w_1stIn", "w_df",
    "loser_name", "loser_rank", "loser_rank_points",
    "l_ace", "l_svpt", "l_1stIn", "l_df",
]
# Note: 'surface' is retained here but not used in the player-level reshape below.
# It could be useful for surface-level analysis if you want to extend this pipeline later.
# TODO: Step 2 - assign the column list to keep_cols_selection
keep_cols_selection = ...
df = run_guarded(
    step_label="JOIN_DATA Exercise 4",
    action=lambda: df[keep_cols_selection].copy(),
    user_error_message="Could not select the required columns. Check keep_cols_selection.",
    logger=logger,
)
print(f"Shape after filtering: {df.shape}")
logger.info("JOIN_DATA Exercise 4 completed with shape %s.", df.shape)

## [ASIDE] Feature selection early

Carrying unnecessary columns through an analysis slows things down and makes DataFrames
harder to read. Selecting only what you need at the point of loading is a habit worth
building — the equivalent of writing a targeted SQL SELECT rather than SELECT *.

## Exercise 5: Build a Player-Centric View

The match data has one row per match with winner-prefixed and loser-prefixed columns.
To aggregate by player, we need to reshape: create one row per player appearance
(win or loss) with consistent column names.

In [ ]:
def build_player_matches(df: pd.DataFrame) -> pd.DataFrame:
    """
    [Write your docstring here]

    Parameters
    ----------
    df : pd.DataFrame
        ...

    Returns
    -------
    pd.DataFrame
        One row per player per match with columns:
        tourney_date, player, rank, rank_points, ace, svpt, first_in, df_count, won, year.
    """
    # Your code here
    pass


player_matches = run_guarded(
    step_label="JOIN_DATA Exercise 5",
    action=lambda: build_player_matches(df),
    user_error_message="Could not build player-level match rows. Check build_player_matches implementation.",
    logger=logger,
)
print(f"Player-match rows: {player_matches.shape}")
player_matches.head()
logger.info("JOIN_DATA Exercise 5 completed with shape %s.", player_matches.shape)

## Exercise 6: Compute Per-Match Rates

Add derived rate columns before aggregating.

In [ ]:
# TODO: set each variable to the correct player-match column name used in rate calculations.
first_in_col = ...
svpt_col = ...
ace_col = ...
df_col = ...

def _exercise_6_rates():
    player_matches["first_serve_pct"] = player_matches[first_in_col] / player_matches[svpt_col]
    player_matches["ace_rate"] = player_matches[ace_col] / player_matches[svpt_col]
    player_matches["df_rate"] = player_matches[df_col] / player_matches[svpt_col]

run_guarded(
    step_label="JOIN_DATA Exercise 6",
    action=_exercise_6_rates,
    user_error_message="Could not compute per-match rates. Check selected columns and division logic.",
    logger=logger,
)
logger.info("JOIN_DATA Exercise 6 completed.")

## Exercise 7: Aggregate to Player-Season Level

In [ ]:
def aggregate_player_season(player_matches: pd.DataFrame) -> pd.DataFrame:
    """
    [Write your docstring here]

    Parameters
    ----------
    player_matches : pd.DataFrame
        ...

    Returns
    -------
    pd.DataFrame
        Columns: player, year, win_count, total_matches, win_rate,
                 avg_first_serve_pct, avg_ace_rate, avg_df_rate, avg_rank.
    """
    # Your code here
    pass


season_stats = run_guarded(
    step_label="JOIN_DATA Exercise 7",
    action=lambda: aggregate_player_season(player_matches),
    user_error_message="Could not aggregate player-season statistics. Check aggregate_player_season implementation.",
    logger=logger,
)
print(f"Player-season rows: {season_stats.shape}")
season_stats.head()
logger.info("JOIN_DATA Exercise 7 completed with shape %s.", season_stats.shape)

## Exercise 8: Season-End Ranking Points

For each player-season, take the ranking points from their last match of the year.
This is a proxy for year-end ranking — not official, but a good approximation.

In [ ]:
def get_season_end_points(player_matches: pd.DataFrame) -> pd.DataFrame:
    """
    [Write your docstring here]

    Parameters
    ----------
    player_matches : pd.DataFrame
        ...

    Returns
    -------
    pd.DataFrame
        Columns: player, year, ranking_points.
    """
    # Your code here
    pass


season_end_points = run_guarded(
    step_label="JOIN_DATA Exercise 8",
    action=lambda: get_season_end_points(player_matches),
    user_error_message="Could not compute season-end ranking points. Check get_season_end_points implementation.",
    logger=logger,
)
season_end_points.head()
logger.info("JOIN_DATA Exercise 8 completed with shape %s.", season_end_points.shape)

## Exercise 9: Merge and Write Output

In [ ]:
# TODO: set merge inputs to combine season stats with season-end points.
merge_fn = ...
merge_keys = ["player", "year"]
merge_how = "inner"

compiled = run_guarded(
    step_label="JOIN_DATA Exercise 9",
    action=lambda: merge_fn(season_end_points, on=merge_keys, how=merge_how),
    user_error_message="Could not merge season_stats with season_end_points. Check merge callable and keys.",
    logger=logger,
)
compiled = compiled.dropna()
print(f"Final compiled shape: {compiled.shape}")
compiled.head()
logger.info("JOIN_DATA Exercise 9 merge completed with shape %s.", compiled.shape)

In [ ]:
import os; os.makedirs("output", exist_ok=True)

# TODO: set output path and whether to write row index when exporting the compiled data.
output_path = ...
write_index = ...
run_guarded(
    step_label="JOIN_DATA Exercise 9 (Write Output)",
    action=lambda: compiled.to_csv(output_path, index=write_index),
    user_error_message="Could not write compiled output. Check output_path and write_index.",
    logger=logger,
)

print(f"Written to {output_path}")
logger.info("JOIN_DATA Exercise 9 write completed: %s", output_path)

## Git Signpost

Commit the notebook. The compiled CSV lives in `output/` which is gitignored — the learner
regenerates it by running this notebook.

```bash
git add JOIN_DATA.ipynb
git commit -m "complete JOIN_DATA notebook"
```